# Phase 3: Model Development and Validation


## 1. Modeling Dataset Preparation

Prepare the final engineered dataset for model development and establish a leakage-free evaluation strategy.


### 1.1 Dataset Loading and Initial Validation

Load the engineered Zillow-FEMA dataset, confirm the expected structure, convert the date column, sort the panel by time and county, and check that the required modeling variables are available.


In [2]:
# ==================================================
# Phase 3: Model Development and Validation
# Step 1.1: Dataset Loading and Initial Validation
# ==================================================

# Import core libraries
import pandas as pd
import numpy as np

# Import modeling utilities
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Import statistical testing
from scipy.stats import ttest_rel

# Optional: suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# --------------------------------------------------
# Define file paths
# --------------------------------------------------

DATA_PATH = "../data/processed/engineered_zillow_fema_data.csv"
RESULTS_TABLES_PATH = "../results/tables/"
RESULTS_MODELS_PATH = "../results/models/"

# --------------------------------------------------
# Load engineered dataset
# --------------------------------------------------

df = pd.read_csv(DATA_PATH)

# Convert Date column to datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Sort panel dataset by time and county
df = df.sort_values(by=["Date", "STCOFIPS"]).reset_index(drop=True)

# --------------------------------------------------
# Initial validation checks
# --------------------------------------------------

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Number of counties: {df['STCOFIPS'].nunique()}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Number of missing values in dataset: {df.isnull().sum().sum()}")

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

Dataset loaded successfully.
Dataset shape: (11963, 25)
Number of counties: 67
Date range: 2011-01-31 to 2025-12-31
Number of missing values in dataset: 2856

Column names:
['RegionID', 'SizeRank', 'RegionName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS', 'Date', 'HousingPrice', 'STCOFIPS', 'STATE', 'STATEABBRV', 'COUNTY', 'POPULATION', 'CFLD_RISKS', 'HRCN_RISKS', 'SOVI_SCORE', 'RESL_SCORE', 'Month', 'TimeIndex', 'is_metro', 'HousingPrice_lag1', 'HousingPrice_lag12', 'CFLD_RISKS_x_Time', 'HRCN_RISKS_x_Time']

First five rows:


,RegionID,SizeRank,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,Date,HousingPrice,STCOFIPS,...,HRCN_RISKS,SOVI_SCORE,RESL_SCORE,Month,TimeIndex,is_metro,HousingPrice_lag1,HousingPrice_lag12,CFLD_RISKS_x_Time,HRCN_RISKS_x_Time
0,1509,251,Alachua County,FL,"Gainesville, FL",12,1,2011-01-31,151065.304144,12001,...,96.704214,34.764631,80.979644,1,12,1,152494.855898,167705.397821,0.0,1160.450563
1,378,1544,Baker County,FL,"Jacksonville, FL",12,3,2011-01-31,131145.170913,12003,...,74.468085,24.427481,71.787532,1,12,1,131761.333829,140451.290090,0.0,893.617021
2,67,371,Bay County,FL,"Panama City, FL",12,5,2011-01-31,164525.255977,12005,...,98.664998,30.852417,29.452926,1,12,1,166058.803964,177323.256436,926.4,1183.979975
3,2148,1545,Bradford County,FL,NaN,12,7,2011-01-31,104394.359137,12007,...,87.567793,48.314249,37.913486,1,12,0,105000.740591,110644.912427,0.0,1050.813517
4,1556,111,Brevard County,FL,"Palm Bay-Melbourne-Titusville, FL",12,9,2011-01-31,122808.989054,12009,...,99.707968,40.839695,63.358779,1,12,1,124197.366085,137329.243830,984.0,1196.495620


In [3]:
# ==================================================
# Validate Missing Values in Modeling Variables
# ==================================================

# Define all variables that may be used in Phase 3 modeling
modeling_columns = [
    "HousingPrice",
    "TimeIndex",
    "Month",
    "SizeRank",
    "is_metro",
    "POPULATION",
    "HousingPrice_lag1",
    "HousingPrice_lag12",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
    "CFLD_RISKS_x_Time",
    "HRCN_RISKS_x_Time"
]

# Check missing values only in modeling-relevant columns
missing_modeling_values = df[modeling_columns].isnull().sum()

print("Missing values in modeling-relevant columns:")
display(missing_modeling_values)

print("\nTotal missing values in modeling-relevant columns:")
print(missing_modeling_values.sum())

Missing values in modeling-relevant columns:


HousingPrice          0
TimeIndex             0
Month                 0
SizeRank              0
is_metro              0
POPULATION            0
HousingPrice_lag1     0
HousingPrice_lag12    0
CFLD_RISKS            0
HRCN_RISKS            0
SOVI_SCORE            0
RESL_SCORE            0
CFLD_RISKS_x_Time     0
HRCN_RISKS_x_Time     0
dtype: int64


Total missing values in modeling-relevant columns:
0


### 1.2 Target and Feature Set Definition

Define the target variable, baseline feature set, and climate-enhanced feature set. The baseline feature set represents a traditional housing valuation model, while the climate-enhanced feature set adds FEMA climate-risk and vulnerability variables.


In [4]:
# ==================================================
# Step 1.2: Target and Feature Set Definition
# ==================================================

# --------------------------------------------------
# Define target variable
# --------------------------------------------------

target = "HousingPrice"

# --------------------------------------------------
# Define baseline feature set
# --------------------------------------------------
# These variables represent a traditional housing-market model
# without explicit climate-risk information.

baseline_features = [
    "TimeIndex",
    "Month",
    "SizeRank",
    "is_metro",
    "POPULATION",
    "HousingPrice_lag1",
    "HousingPrice_lag12"
]

# --------------------------------------------------
# Define climate feature set
# --------------------------------------------------
# These variables add FEMA climate-risk, vulnerability,
# resilience, and risk-time interaction information.

climate_features = [
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
    "CFLD_RISKS_x_Time",
    "HRCN_RISKS_x_Time"
]

# --------------------------------------------------
# Define climate-enhanced feature set
# --------------------------------------------------

climate_enhanced_features = baseline_features + climate_features

# --------------------------------------------------
# Validate feature sets
# --------------------------------------------------

print("Target variable:")
print(target)

print("\nBaseline feature set:")
print(baseline_features)
print(f"Number of baseline features: {len(baseline_features)}")

print("\nClimate features added:")
print(climate_features)
print(f"Number of climate features added: {len(climate_features)}")

print("\nClimate-enhanced feature set:")
print(climate_enhanced_features)
print(f"Number of climate-enhanced features: {len(climate_enhanced_features)}")

# Confirm all selected columns exist in the dataset
required_columns = [target] + climate_enhanced_features
missing_columns = [col for col in required_columns if col not in df.columns]

print("\nMissing required columns:")
print(missing_columns)

Target variable:
HousingPrice

Baseline feature set:
['TimeIndex', 'Month', 'SizeRank', 'is_metro', 'POPULATION', 'HousingPrice_lag1', 'HousingPrice_lag12']
Number of baseline features: 7

Climate features added:
['CFLD_RISKS', 'HRCN_RISKS', 'SOVI_SCORE', 'RESL_SCORE', 'CFLD_RISKS_x_Time', 'HRCN_RISKS_x_Time']
Number of climate features added: 6

Climate-enhanced feature set:
['TimeIndex', 'Month', 'SizeRank', 'is_metro', 'POPULATION', 'HousingPrice_lag1', 'HousingPrice_lag12', 'CFLD_RISKS', 'HRCN_RISKS', 'SOVI_SCORE', 'RESL_SCORE', 'CFLD_RISKS_x_Time', 'HRCN_RISKS_x_Time']
Number of climate-enhanced features: 13

Missing required columns:
[]


### 1.3 Time-Based Train/Test Split and Validation Strategy

Create the final training and test datasets using a time-based split. The training period covers 2011–2022, while the holdout test period covers 2023–2025. TimeSeriesSplit will be used within the training period for validation and hyperparameter tuning.


In [5]:
# ==================================================
# Step 1.3: Time-Based Train/Test Split
# ==================================================

# --------------------------------------------------
# Define train and test periods
# --------------------------------------------------

train_end_date = "2022-12-31"
test_start_date = "2023-01-01"

# --------------------------------------------------
# Create time-based train and test datasets
# --------------------------------------------------

train_df = df[df["Date"] <= train_end_date].copy()
test_df = df[df["Date"] >= test_start_date].copy()

# --------------------------------------------------
# Create baseline feature matrices
# --------------------------------------------------

X_train_baseline = train_df[baseline_features]
X_test_baseline = test_df[baseline_features]

# --------------------------------------------------
# Create climate-enhanced feature matrices
# --------------------------------------------------

X_train_climate = train_df[climate_enhanced_features]
X_test_climate = test_df[climate_enhanced_features]

# --------------------------------------------------
# Create target vectors
# --------------------------------------------------

y_train = train_df[target]
y_test = test_df[target]

# --------------------------------------------------
# Validate split results
# --------------------------------------------------

print("Time-based train/test split completed successfully.")

print("\nTraining set:")
print(f"Date range: {train_df['Date'].min().date()} to {train_df['Date'].max().date()}")
print(f"Observations: {train_df.shape[0]}")
print(f"Counties: {train_df['STCOFIPS'].nunique()}")

print("\nTest set:")
print(f"Date range: {test_df['Date'].min().date()} to {test_df['Date'].max().date()}")
print(f"Observations: {test_df.shape[0]}")
print(f"Counties: {test_df['STCOFIPS'].nunique()}")

print("\nFeature matrix shapes:")
print(f"X_train_baseline: {X_train_baseline.shape}")
print(f"X_test_baseline: {X_test_baseline.shape}")
print(f"X_train_climate: {X_train_climate.shape}")
print(f"X_test_climate: {X_test_climate.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

Time-based train/test split completed successfully.

Training set:
Date range: 2011-01-31 to 2022-12-31
Observations: 9551
Counties: 67

Test set:
Date range: 2023-01-31 to 2025-12-31
Observations: 2412
Counties: 67

Feature matrix shapes:
X_train_baseline: (9551, 7)
X_test_baseline: (2412, 7)
X_train_climate: (9551, 13)
X_test_climate: (2412, 13)
y_train: (9551,)
y_test: (2412,)


In [6]:
# ==================================================
# Step 1.3: TimeSeriesSplit Validation Strategy
# ==================================================

# --------------------------------------------------
# Define time series cross-validation strategy
# --------------------------------------------------
# TimeSeriesSplit preserves temporal order by ensuring that
# validation data always comes after training data.

n_splits = 5

tscv = TimeSeriesSplit(n_splits=n_splits)

# --------------------------------------------------
# Validate TimeSeriesSplit structure
# --------------------------------------------------

print("TimeSeriesSplit validation strategy created successfully.")
print(f"Number of splits: {n_splits}")

print("\nFold structure:")

for fold, (train_index, val_index) in enumerate(tscv.split(X_train_baseline), start=1):
    fold_train_dates = train_df.iloc[train_index]["Date"]
    fold_val_dates = train_df.iloc[val_index]["Date"]
    
    print(f"\nFold {fold}")
    print(f"Training observations: {len(train_index)}")
    print(f"Validation observations: {len(val_index)}")
    print(f"Training date range: {fold_train_dates.min().date()} to {fold_train_dates.max().date()}")
    print(f"Validation date range: {fold_val_dates.min().date()} to {fold_val_dates.max().date()}")

TimeSeriesSplit validation strategy created successfully.
Number of splits: 5

Fold structure:

Fold 1
Training observations: 1596
Validation observations: 1591
Training date range: 2011-01-31 to 2013-01-31
Validation date range: 2013-01-31 to 2015-01-31

Fold 2
Training observations: 3187
Validation observations: 1591
Training date range: 2011-01-31 to 2015-01-31
Validation date range: 2015-01-31 to 2017-01-31

Fold 3
Training observations: 4778
Validation observations: 1591
Training date range: 2011-01-31 to 2017-01-31
Validation date range: 2017-01-31 to 2019-01-31

Fold 4
Training observations: 6369
Validation observations: 1591
Training date range: 2011-01-31 to 2019-01-31
Validation date range: 2019-01-31 to 2021-01-31

Fold 5
Training observations: 7960
Validation observations: 1591
Training date range: 2011-01-31 to 2021-01-31
Validation date range: 2021-01-31 to 2022-12-31


### 1.4 Methodology Split Summary

Summarize the modeling dataset, training period, test period, observation counts, and validation strategy in a report-ready table.


### Step 1 Key Takeaways

- The modeling dataset is structured as a county-month panel.
- The target variable is housing price.
- Baseline and climate-enhanced feature sets are clearly separated.
- A time-based split is used to avoid future leakage.
- TimeSeriesSplit is used only within the training period.


## 2. Baseline Model Development

Estimate housing valuation models using only traditional housing-market, temporal, population, and lagged-price features.


### 2.1 Baseline Linear Regression Model

Train the baseline OLS model using the baseline feature set. This model serves as the academic benchmark for comparison.


In [7]:
# ==================================================
# Step 2: Baseline Model Development
# Evaluation Function
# ==================================================

def evaluate_predictions(y_true, y_pred):
    """
    Evaluate model predictions using standard regression metrics.

    Parameters
    ----------
    y_true : array-like
        Actual target values.
    y_pred : array-like
        Predicted target values.

    Returns
    -------
    dict
        Dictionary containing RMSE, MAE, and R².
    """
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


# Quick function test using a small example
test_metrics = evaluate_predictions(
    y_true=[100, 200, 300],
    y_pred=[110, 190, 310]
)

print("Evaluation function created successfully.")
print(test_metrics)

Evaluation function created successfully.
{'RMSE': np.float64(10.0), 'MAE': 10.0, 'R2': 0.985}


In [8]:
# ==================================================
# Step 2.1: Baseline Linear Regression Model
# ==================================================

# --------------------------------------------------
# Initialize and train baseline linear regression model
# --------------------------------------------------

baseline_lr_model = LinearRegression()

baseline_lr_model.fit(X_train_baseline, y_train)

# --------------------------------------------------
# Generate predictions on the holdout test set
# --------------------------------------------------

baseline_lr_predictions = baseline_lr_model.predict(X_test_baseline)

# --------------------------------------------------
# Evaluate model performance
# --------------------------------------------------

baseline_lr_metrics = evaluate_predictions(
    y_true=y_test,
    y_pred=baseline_lr_predictions
)

# --------------------------------------------------
# Store model results
# --------------------------------------------------

baseline_lr_results = {
    "Model": "Linear Regression",
    "Feature_Set": "Baseline",
    "RMSE": baseline_lr_metrics["RMSE"],
    "MAE": baseline_lr_metrics["MAE"],
    "R2": baseline_lr_metrics["R2"]
}

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Baseline Linear Regression model trained successfully.")
print("\nBaseline Linear Regression Test Performance:")
print(f"RMSE: {baseline_lr_results['RMSE']:.2f}")
print(f"MAE: {baseline_lr_results['MAE']:.2f}")
print(f"R²: {baseline_lr_results['R2']:.4f}")

Baseline Linear Regression model trained successfully.

Baseline Linear Regression Test Performance:
RMSE: 2165.74
MAE: 1601.21
R²: 0.9997


### 2.2 Baseline Random Forest Model

Train and tune the baseline Random Forest model using TimeSeriesSplit. This model captures nonlinear relationships without using climate-risk variables.


In [9]:
# ==================================================
# Step 2.2: Baseline Random Forest Model
# ==================================================

from sklearn.model_selection import GridSearchCV

# --------------------------------------------------
# Define baseline Random Forest model
# --------------------------------------------------

baseline_rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# --------------------------------------------------
# Define hyperparameter grid
# --------------------------------------------------
# A small grid is used to keep tuning computationally manageable
# while still allowing the model to test different levels of complexity.

rf_param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 3, 5]
}

# --------------------------------------------------
# Set up TimeSeriesSplit grid search
# --------------------------------------------------

baseline_rf_grid = GridSearchCV(
    estimator=baseline_rf,
    param_grid=rf_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# Fit Random Forest using baseline features
# --------------------------------------------------

baseline_rf_grid.fit(X_train_baseline, y_train)

# --------------------------------------------------
# Select best model and generate test predictions
# --------------------------------------------------

baseline_rf_best_model = baseline_rf_grid.best_estimator_

baseline_rf_predictions = baseline_rf_best_model.predict(X_test_baseline)

# --------------------------------------------------
# Evaluate model performance
# --------------------------------------------------

baseline_rf_metrics = evaluate_predictions(
    y_true=y_test,
    y_pred=baseline_rf_predictions
)

# --------------------------------------------------
# Store model results
# --------------------------------------------------

baseline_rf_results = {
    "Model": "Random Forest",
    "Feature_Set": "Baseline",
    "Best_Params": baseline_rf_grid.best_params_,
    "CV_RMSE": -baseline_rf_grid.best_score_,
    "RMSE": baseline_rf_metrics["RMSE"],
    "MAE": baseline_rf_metrics["MAE"],
    "R2": baseline_rf_metrics["R2"]
}

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Baseline Random Forest model trained successfully.")

print("\nBest Random Forest Parameters:")
print(baseline_rf_results["Best_Params"])

print("\nCross-Validation Performance:")
print(f"Best CV RMSE: {baseline_rf_results['CV_RMSE']:.2f}")

print("\nBaseline Random Forest Test Performance:")
print(f"RMSE: {baseline_rf_results['RMSE']:.2f}")
print(f"MAE: {baseline_rf_results['MAE']:.2f}")
print(f"R²: {baseline_rf_results['R2']:.4f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Baseline Random Forest model trained successfully.

Best Random Forest Parameters:
{'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}

Cross-Validation Performance:
Best CV RMSE: 14594.56

Baseline Random Forest Test Performance:
RMSE: 15003.20
MAE: 5107.06
R²: 0.9868


### 2.3 Baseline XGBoost Model

Train and tune the baseline XGBoost model using TimeSeriesSplit. This model serves as a strong tabular-data benchmark without climate-risk variables.


In [10]:
# ==================================================
# Step 2.3: Baseline XGBoost Model
# ==================================================

from xgboost import XGBRegressor

# --------------------------------------------------
# Define baseline XGBoost model
# --------------------------------------------------

baseline_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# --------------------------------------------------
# Define hyperparameter grid
# --------------------------------------------------
# A small grid is used to keep tuning manageable while still
# allowing the model to test different levels of complexity.

xgb_param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0]
}

# --------------------------------------------------
# Set up TimeSeriesSplit grid search
# --------------------------------------------------

baseline_xgb_grid = GridSearchCV(
    estimator=baseline_xgb,
    param_grid=xgb_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# Fit XGBoost using baseline features
# --------------------------------------------------

baseline_xgb_grid.fit(X_train_baseline, y_train)

# --------------------------------------------------
# Select best model and generate test predictions
# --------------------------------------------------

baseline_xgb_best_model = baseline_xgb_grid.best_estimator_

baseline_xgb_predictions = baseline_xgb_best_model.predict(X_test_baseline)

# --------------------------------------------------
# Evaluate model performance
# --------------------------------------------------

baseline_xgb_metrics = evaluate_predictions(
    y_true=y_test,
    y_pred=baseline_xgb_predictions
)

# --------------------------------------------------
# Store model results
# --------------------------------------------------

baseline_xgb_results = {
    "Model": "XGBoost",
    "Feature_Set": "Baseline",
    "Best_Params": baseline_xgb_grid.best_params_,
    "CV_RMSE": -baseline_xgb_grid.best_score_,
    "RMSE": baseline_xgb_metrics["RMSE"],
    "MAE": baseline_xgb_metrics["MAE"],
    "R2": baseline_xgb_metrics["R2"]
}

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Baseline XGBoost model trained successfully.")

print("\nBest XGBoost Parameters:")
print(baseline_xgb_results["Best_Params"])

print("\nCross-Validation Performance:")
print(f"Best CV RMSE: {baseline_xgb_results['CV_RMSE']:.2f}")

print("\nBaseline XGBoost Test Performance:")
print(f"RMSE: {baseline_xgb_results['RMSE']:.2f}")
print(f"MAE: {baseline_xgb_results['MAE']:.2f}")
print(f"R²: {baseline_xgb_results['R2']:.4f}")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Baseline XGBoost model trained successfully.

Best XGBoost Parameters:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}

Cross-Validation Performance:
Best CV RMSE: 15091.34

Baseline XGBoost Test Performance:
RMSE: 30849.08
MAE: 9252.48
R²: 0.9440


### 2.4 Baseline Model Results Summary

Summarize baseline model performance using RMSE, MAE, and R² on the 2023–2025 holdout test set.


In [13]:
# ==================================================
# Step 2.4: Baseline Model Results Summary
# ==================================================

# --------------------------------------------------
# Create clean baseline model performance table
# --------------------------------------------------

baseline_results_df = pd.DataFrame([
    {
        "Model": baseline_lr_results["Model"],
        "Feature Set": baseline_lr_results["Feature_Set"],
        "RMSE": baseline_lr_results["RMSE"],
        "MAE": baseline_lr_results["MAE"],
        "R²": baseline_lr_results["R2"]
    },
    {
        "Model": baseline_rf_results["Model"],
        "Feature Set": baseline_rf_results["Feature_Set"],
        "RMSE": baseline_rf_results["RMSE"],
        "MAE": baseline_rf_results["MAE"],
        "R²": baseline_rf_results["R2"]
    },
    {
        "Model": baseline_xgb_results["Model"],
        "Feature Set": baseline_xgb_results["Feature_Set"],
        "RMSE": baseline_xgb_results["RMSE"],
        "MAE": baseline_xgb_results["MAE"],
        "R²": baseline_xgb_results["R2"]
    }
])

# --------------------------------------------------
# Round values for cleaner display
# --------------------------------------------------

baseline_results_display = baseline_results_df.copy()

baseline_results_display["RMSE"] = baseline_results_display["RMSE"].round(2)
baseline_results_display["MAE"] = baseline_results_display["MAE"].round(2)
baseline_results_display["R²"] = baseline_results_display["R²"].round(4)

# --------------------------------------------------
# Save baseline results table
# --------------------------------------------------

baseline_results_path = RESULTS_TABLES_PATH + "baseline_model_results.csv"

baseline_results_df.to_csv(baseline_results_path, index=False)

# --------------------------------------------------
# Display baseline results
# --------------------------------------------------

print("Baseline model results table created successfully.")
print(f"Saved to: {baseline_results_path}")

display(baseline_results_display)

Baseline model results table created successfully.
Saved to: ../results/tables/baseline_model_results.csv


,Model,Feature Set,RMSE,MAE,R²
0,Linear Regression,Baseline,2165.74,1601.21,0.9997
1,Random Forest,Baseline,15003.20,5107.06,0.9868
2,XGBoost,Baseline,30849.08,9252.48,0.9440


### Step 2 Key Takeaways

* Baseline models were developed using only traditional housing valuation features, without climate-risk variables.
* All baseline models were trained on the 2011–2022 training period and evaluated on the same 2023–2025 holdout test period.
* Linear Regression produced the strongest baseline performance, with the lowest RMSE and MAE and the highest R².
* Random Forest and XGBoost also achieved strong predictive performance, but they did not outperform the baseline Linear Regression model.
* The strong Linear Regression result suggests that lagged housing prices are highly predictive and may dominate the modeling task.
* This indicates that Florida county-level housing prices exhibit strong temporal persistence.
* These baseline results will serve as the comparison point for the climate-enhanced models in Step 3.
* The key question moving forward is whether adding climate-risk variables improves each model relative to its own baseline version.


## 3. Climate-Enhanced Model Development

Estimate the same model families after adding FEMA climate-risk, vulnerability, resilience, and climate-time interaction variables.


### 3.1 Climate-Enhanced Linear Regression Model

Train the OLS model using the climate-enhanced feature set. This tests whether climate-risk variables improve the linear benchmark.


In [11]:
# ==================================================
# Step 3.1: Climate-Enhanced Linear Regression Model
# ==================================================

# --------------------------------------------------
# Initialize and train climate-enhanced linear regression model
# --------------------------------------------------

climate_lr_model = LinearRegression()

climate_lr_model.fit(X_train_climate, y_train)

# --------------------------------------------------
# Generate predictions on the holdout test set
# --------------------------------------------------

climate_lr_predictions = climate_lr_model.predict(X_test_climate)

# --------------------------------------------------
# Evaluate model performance
# --------------------------------------------------

climate_lr_metrics = evaluate_predictions(
    y_true=y_test,
    y_pred=climate_lr_predictions
)

# --------------------------------------------------
# Store model results
# --------------------------------------------------

climate_lr_results = {
    "Model": "Linear Regression",
    "Feature_Set": "Climate-Enhanced",
    "RMSE": climate_lr_metrics["RMSE"],
    "MAE": climate_lr_metrics["MAE"],
    "R2": climate_lr_metrics["R2"]
}

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Climate-Enhanced Linear Regression model trained successfully.")

print("\nClimate-Enhanced Linear Regression Test Performance:")
print(f"RMSE: {climate_lr_results['RMSE']:.2f}")
print(f"MAE: {climate_lr_results['MAE']:.2f}")
print(f"R²: {climate_lr_results['R2']:.4f}")

Climate-Enhanced Linear Regression model trained successfully.

Climate-Enhanced Linear Regression Test Performance:
RMSE: 2148.55
MAE: 1578.41
R²: 0.9997


### 3.2 Climate-Enhanced Random Forest Model

Train and tune the climate-enhanced Random Forest model using the same TimeSeriesSplit validation strategy as the baseline version.


In [14]:
# ==================================================
# Step 3.2: Climate-Enhanced Random Forest Model
# ==================================================

# --------------------------------------------------
# Define climate-enhanced Random Forest model
# --------------------------------------------------

climate_rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# --------------------------------------------------
# Set up TimeSeriesSplit grid search
# --------------------------------------------------
# The same hyperparameter grid and validation strategy used
# for the baseline Random Forest model are reused here to
# ensure a fair comparison.

climate_rf_grid = GridSearchCV(
    estimator=climate_rf,
    param_grid=rf_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# Fit Random Forest using climate-enhanced features
# --------------------------------------------------

climate_rf_grid.fit(X_train_climate, y_train)

# --------------------------------------------------
# Select best model and generate test predictions
# --------------------------------------------------

climate_rf_best_model = climate_rf_grid.best_estimator_

climate_rf_predictions = climate_rf_best_model.predict(X_test_climate)

# --------------------------------------------------
# Evaluate model performance
# --------------------------------------------------

climate_rf_metrics = evaluate_predictions(
    y_true=y_test,
    y_pred=climate_rf_predictions
)

# --------------------------------------------------
# Store model results
# --------------------------------------------------

climate_rf_results = {
    "Model": "Random Forest",
    "Feature_Set": "Climate-Enhanced",
    "Best_Params": climate_rf_grid.best_params_,
    "CV_RMSE": -climate_rf_grid.best_score_,
    "RMSE": climate_rf_metrics["RMSE"],
    "MAE": climate_rf_metrics["MAE"],
    "R2": climate_rf_metrics["R2"]
}

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Climate-Enhanced Random Forest model trained successfully.")

print("\nBest Random Forest Parameters:")
print(climate_rf_results["Best_Params"])

print("\nCross-Validation Performance:")
print(f"Best CV RMSE: {climate_rf_results['CV_RMSE']:.2f}")

print("\nClimate-Enhanced Random Forest Test Performance:")
print(f"RMSE: {climate_rf_results['RMSE']:.2f}")
print(f"MAE: {climate_rf_results['MAE']:.2f}")
print(f"R²: {climate_rf_results['R2']:.4f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Climate-Enhanced Random Forest model trained successfully.

Best Random Forest Parameters:
{'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 200}

Cross-Validation Performance:
Best CV RMSE: 15302.63

Climate-Enhanced Random Forest Test Performance:
RMSE: 12535.01
MAE: 5283.10
R²: 0.9908


### 3.3 Climate-Enhanced XGBoost Model

Train and tune the climate-enhanced XGBoost model using the same TimeSeriesSplit validation strategy as the baseline version.


In [15]:
# ==================================================
# Step 3.3: Climate-Enhanced XGBoost Model
# ==================================================

# --------------------------------------------------
# Define climate-enhanced XGBoost model
# --------------------------------------------------

climate_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# --------------------------------------------------
# Set up TimeSeriesSplit grid search
# --------------------------------------------------
# The same hyperparameter grid and validation strategy used
# for the baseline XGBoost model are reused here to ensure
# a fair comparison.

climate_xgb_grid = GridSearchCV(
    estimator=climate_xgb,
    param_grid=xgb_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# Fit XGBoost using climate-enhanced features
# --------------------------------------------------

climate_xgb_grid.fit(X_train_climate, y_train)

# --------------------------------------------------
# Select best model and generate test predictions
# --------------------------------------------------

climate_xgb_best_model = climate_xgb_grid.best_estimator_

climate_xgb_predictions = climate_xgb_best_model.predict(X_test_climate)

# --------------------------------------------------
# Evaluate model performance
# --------------------------------------------------

climate_xgb_metrics = evaluate_predictions(
    y_true=y_test,
    y_pred=climate_xgb_predictions
)

# --------------------------------------------------
# Store model results
# --------------------------------------------------

climate_xgb_results = {
    "Model": "XGBoost",
    "Feature_Set": "Climate-Enhanced",
    "Best_Params": climate_xgb_grid.best_params_,
    "CV_RMSE": -climate_xgb_grid.best_score_,
    "RMSE": climate_xgb_metrics["RMSE"],
    "MAE": climate_xgb_metrics["MAE"],
    "R2": climate_xgb_metrics["R2"]
}

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Climate-Enhanced XGBoost model trained successfully.")

print("\nBest XGBoost Parameters:")
print(climate_xgb_results["Best_Params"])

print("\nCross-Validation Performance:")
print(f"Best CV RMSE: {climate_xgb_results['CV_RMSE']:.2f}")

print("\nClimate-Enhanced XGBoost Test Performance:")
print(f"RMSE: {climate_xgb_results['RMSE']:.2f}")
print(f"MAE: {climate_xgb_results['MAE']:.2f}")
print(f"R²: {climate_xgb_results['R2']:.4f}")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Climate-Enhanced XGBoost model trained successfully.

Best XGBoost Parameters:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}

Cross-Validation Performance:
Best CV RMSE: 17052.45

Climate-Enhanced XGBoost Test Performance:
RMSE: 41192.89
MAE: 13214.53
R²: 0.9002


### 3.4 Climate-Enhanced Model Results Summary

Summarize climate-enhanced model performance using RMSE, MAE, and R² on the 2023–2025 holdout test set.


In [16]:
# ==================================================
# Step 3.4: Climate-Enhanced Model Results Summary
# ==================================================

# --------------------------------------------------
# Create clean climate-enhanced model performance table
# --------------------------------------------------

climate_results_df = pd.DataFrame([
    {
        "Model": climate_lr_results["Model"],
        "Feature Set": climate_lr_results["Feature_Set"],
        "RMSE": climate_lr_results["RMSE"],
        "MAE": climate_lr_results["MAE"],
        "R²": climate_lr_results["R2"]
    },
    {
        "Model": climate_rf_results["Model"],
        "Feature Set": climate_rf_results["Feature_Set"],
        "RMSE": climate_rf_results["RMSE"],
        "MAE": climate_rf_results["MAE"],
        "R²": climate_rf_results["R2"]
    },
    {
        "Model": climate_xgb_results["Model"],
        "Feature Set": climate_xgb_results["Feature_Set"],
        "RMSE": climate_xgb_results["RMSE"],
        "MAE": climate_xgb_results["MAE"],
        "R²": climate_xgb_results["R2"]
    }
])

# --------------------------------------------------
# Round values for cleaner display
# --------------------------------------------------

climate_results_display = climate_results_df.copy()

climate_results_display["RMSE"] = climate_results_display["RMSE"].round(2)
climate_results_display["MAE"] = climate_results_display["MAE"].round(2)
climate_results_display["R²"] = climate_results_display["R²"].round(4)

# --------------------------------------------------
# Save climate-enhanced results table
# --------------------------------------------------

climate_results_path = RESULTS_TABLES_PATH + "climate_model_results.csv"

climate_results_df.to_csv(climate_results_path, index=False)

# --------------------------------------------------
# Display climate-enhanced results
# --------------------------------------------------

print("Climate-enhanced model results table created successfully.")
print(f"Saved to: {climate_results_path}")

display(climate_results_display)

Climate-enhanced model results table created successfully.
Saved to: ../results/tables/climate_model_results.csv


,Model,Feature Set,RMSE,MAE,R²
0,Linear Regression,Climate-Enhanced,2148.55,1578.41,0.9997
1,Random Forest,Climate-Enhanced,12535.01,5283.10,0.9908
2,XGBoost,Climate-Enhanced,41192.89,13214.53,0.9002


### Step 3 Key Takeaways

* Climate-enhanced models were developed using the same train/test split, validation strategy, and evaluation metrics as the baseline models.
* The only major change from Step 2 was the inclusion of climate-risk and related variables in the feature set.
* The climate-enhanced Linear Regression model produced a small improvement in RMSE and MAE compared with the baseline Linear Regression model.
* The climate-enhanced Random Forest model improved RMSE and R² compared with the baseline Random Forest model, although MAE increased slightly.
* The climate-enhanced XGBoost model performed worse than the baseline XGBoost model across RMSE, MAE, and R².
* These results suggest that the contribution of climate-risk variables is model-dependent.
* Climate-risk variables appear to provide some incremental predictive value for Linear Regression and Random Forest, but they did not improve XGBoost performance under the current model specification.
* The next step will directly compare baseline and climate-enhanced models to evaluate whether climate-risk variables improve housing valuation performance overall.


## 4. Performance Comparison

Compare baseline and climate-enhanced models to determine whether climate-risk variables improve housing valuation performance.


### 4.1 Master Results Table

Combine baseline and climate-enhanced results into one table showing model type, climate-variable inclusion, RMSE, MAE, and R².


### 4.2 Performance Improvement Summary

Calculate the change in RMSE, MAE, and R² between each baseline model and its climate-enhanced version.


### Step 4 Key Takeaways

- The performance comparison directly answers the main research question.
- Improvements should be assessed within the same model family.
- RMSE and MAE improvements are interpreted in dollar terms.


## 5. Statistical Validation

Test whether the performance differences between baseline and climate-enhanced models are statistically meaningful.


### 5.1 Prediction Error Preparation

Create matched prediction-error vectors for each baseline and climate-enhanced model pair using the same 2023–2025 test observations.


In [17]:
# ==================================================
# Step 5.1: Prediction Error Preparation
# ==================================================

# --------------------------------------------------
# Create prediction error dataframe for test period
# --------------------------------------------------
# Absolute errors are calculated for each model using
# the same 2023–2025 holdout test observations.

prediction_errors_df = pd.DataFrame({
    "Date": test_df["Date"].values,
    "STCOFIPS": test_df["STCOFIPS"].values,
    "RegionName": test_df["RegionName"].values,
    "Actual_HousingPrice": y_test.values,
    
    # Linear Regression absolute errors
    "Baseline_LR_Error": np.abs(y_test.values - baseline_lr_predictions),
    "Climate_LR_Error": np.abs(y_test.values - climate_lr_predictions),
    
    # Random Forest absolute errors
    "Baseline_RF_Error": np.abs(y_test.values - baseline_rf_predictions),
    "Climate_RF_Error": np.abs(y_test.values - climate_rf_predictions),
    
    # XGBoost absolute errors
    "Baseline_XGB_Error": np.abs(y_test.values - baseline_xgb_predictions),
    "Climate_XGB_Error": np.abs(y_test.values - climate_xgb_predictions)
})

# --------------------------------------------------
# Validate prediction error dataframe
# --------------------------------------------------

print("Prediction error dataframe created successfully.")
print(f"Shape: {prediction_errors_df.shape}")
print(f"Date range: {prediction_errors_df['Date'].min().date()} to {prediction_errors_df['Date'].max().date()}")
print(f"Number of counties: {prediction_errors_df['STCOFIPS'].nunique()}")

print("\nMissing values in prediction error dataframe:")
display(prediction_errors_df.isnull().sum())

print("\nPreview of prediction error dataframe:")
display(prediction_errors_df.head())

Prediction error dataframe created successfully.
Shape: (2412, 10)
Date range: 2023-01-31 to 2025-12-31
Number of counties: 67

Missing values in prediction error dataframe:


Date                   0
STCOFIPS               0
RegionName             0
Actual_HousingPrice    0
Baseline_LR_Error      0
Climate_LR_Error       0
Baseline_RF_Error      0
Climate_RF_Error       0
Baseline_XGB_Error     0
Climate_XGB_Error      0
dtype: int64


Preview of prediction error dataframe:


,Date,STCOFIPS,RegionName,Actual_HousingPrice,Baseline_LR_Error,Climate_LR_Error,Baseline_RF_Error,Climate_RF_Error,Baseline_XGB_Error,Climate_XGB_Error
0,2023-01-31,12001,Alachua County,290215.981638,3818.829364,3749.560245,1095.240257,1069.110020,779.143362,2448.737112
1,2023-01-31,12003,Baker County,294970.834187,4479.926750,4051.678453,3506.415050,3350.125580,1473.322063,5108.353313
2,2023-01-31,12005,Bay County,357327.930503,4390.112822,4472.419813,860.857631,62.578095,1739.788247,614.631997
3,2023-01-31,12007,Bradford County,224142.412400,4094.209216,3893.330946,3367.409086,3949.944970,2696.493850,607.884475
4,2023-01-31,12009,Brevard County,352551.504030,6626.372978,6815.599876,3975.017143,3988.120090,2083.683470,1614.589720


### 5.2 Paired t-Test Results

Conduct paired t-tests comparing baseline and climate-enhanced absolute prediction errors for OLS, Random Forest, and XGBoost.


In [18]:
# ==================================================
# Step 5.2: Paired t-Test Results
# ==================================================

# --------------------------------------------------
# Define paired t-test helper function
# --------------------------------------------------

def run_paired_ttest(model_name, baseline_error_col, climate_error_col):
    """
    Run a one-sided paired t-test comparing baseline and climate-enhanced
    absolute prediction errors for the same test observations.

    The test evaluates whether baseline errors are significantly greater
    than climate-enhanced errors.

    Parameters
    ----------
    model_name : str
        Name of the model being compared.
    baseline_error_col : str
        Column name containing baseline model absolute errors.
    climate_error_col : str
        Column name containing climate-enhanced model absolute errors.

    Returns
    -------
    dict
        Summary of paired t-test results.
    """
    
    baseline_errors = prediction_errors_df[baseline_error_col]
    climate_errors = prediction_errors_df[climate_error_col]
    
    # One-sided paired t-test:
    # H1: baseline errors are greater than climate-enhanced errors
    t_stat, p_value = ttest_rel(
        baseline_errors,
        climate_errors,
        alternative="greater"
    )
    
    mean_baseline_error = baseline_errors.mean()
    mean_climate_error = climate_errors.mean()
    mean_difference = mean_baseline_error - mean_climate_error
    
    if p_value < 0.05 and mean_difference > 0:
        conclusion = "Significant improvement"
    elif mean_difference > 0:
        conclusion = "Improvement not statistically significant"
    else:
        conclusion = "No improvement"
    
    return {
        "Model": model_name,
        "Mean Baseline Error": mean_baseline_error,
        "Mean Climate Error": mean_climate_error,
        "Mean Difference": mean_difference,
        "t-statistic": t_stat,
        "p-value": p_value,
        "Significant at 5%": p_value < 0.05,
        "Conclusion": conclusion
    }


# --------------------------------------------------
# Run paired t-tests for each model family
# --------------------------------------------------

ttest_results = [
    run_paired_ttest(
        model_name="Linear Regression",
        baseline_error_col="Baseline_LR_Error",
        climate_error_col="Climate_LR_Error"
    ),
    run_paired_ttest(
        model_name="Random Forest",
        baseline_error_col="Baseline_RF_Error",
        climate_error_col="Climate_RF_Error"
    ),
    run_paired_ttest(
        model_name="XGBoost",
        baseline_error_col="Baseline_XGB_Error",
        climate_error_col="Climate_XGB_Error"
    )
]

ttest_results_df = pd.DataFrame(ttest_results)

# --------------------------------------------------
# Round values for cleaner display
# --------------------------------------------------

ttest_results_display = ttest_results_df.copy()

columns_to_round = [
    "Mean Baseline Error",
    "Mean Climate Error",
    "Mean Difference",
    "t-statistic",
    "p-value"
]

for col in columns_to_round:
    ttest_results_display[col] = ttest_results_display[col].round(4)

# --------------------------------------------------
# Save statistical testing results
# --------------------------------------------------

ttest_results_path = RESULTS_TABLES_PATH + "statistical_testing_results.csv"

ttest_results_df.to_csv(ttest_results_path, index=False)

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Paired t-test results created successfully.")
print(f"Saved to: {ttest_results_path}")

display(ttest_results_display)

Paired t-test results created successfully.
Saved to: ../results/tables/statistical_testing_results.csv


,Model,Mean Baseline Error,Mean Climate Error,Mean Difference,t-statistic,p-value,Significant at 5%,Conclusion
0,Linear Regression,1601.2110,1578.4085,22.8025,6.3695,0.000,True,Significant improvement
1,Random Forest,5107.0629,5283.0964,-176.0335,-2.6518,0.996,False,No improvement
2,XGBoost,9252.4780,13214.5321,-3962.0541,-11.5554,1.000,False,No improvement


### 5.3 Statistical Testing Summary

Summarize p-values and significance decisions in a report-ready statistical testing table.


In [19]:
# ==================================================
# Step 5.3: Statistical Testing Summary
# ==================================================

# --------------------------------------------------
# Create report-ready statistical testing table
# --------------------------------------------------

statistical_summary_df = ttest_results_df.copy()

# Rename columns for cleaner reporting
statistical_summary_df = statistical_summary_df.rename(columns={
    "Mean Baseline Error": "Baseline MAE",
    "Mean Climate Error": "Climate-Enhanced MAE",
    "Mean Difference": "MAE Difference",
    "p-value": "p-value",
    "Significant at 5%": "Significant at 5%"
})

# Keep only report-relevant columns
statistical_summary_df = statistical_summary_df[
    [
        "Model",
        "Baseline MAE",
        "Climate-Enhanced MAE",
        "MAE Difference",
        "p-value",
        "Significant at 5%",
        "Conclusion"
    ]
]

# --------------------------------------------------
# Round numeric values for cleaner display
# --------------------------------------------------

statistical_summary_display = statistical_summary_df.copy()

statistical_summary_display["Baseline MAE"] = statistical_summary_display["Baseline MAE"].round(2)
statistical_summary_display["Climate-Enhanced MAE"] = statistical_summary_display["Climate-Enhanced MAE"].round(2)
statistical_summary_display["MAE Difference"] = statistical_summary_display["MAE Difference"].round(2)
statistical_summary_display["p-value"] = statistical_summary_display["p-value"].round(4)

# --------------------------------------------------
# Save report-ready statistical testing summary
# --------------------------------------------------

statistical_summary_path = RESULTS_TABLES_PATH + "statistical_testing_summary.csv"

statistical_summary_df.to_csv(statistical_summary_path, index=False)

# --------------------------------------------------
# Display statistical testing summary
# --------------------------------------------------

print("Statistical testing summary table created successfully.")
print(f"Saved to: {statistical_summary_path}")

display(statistical_summary_display)

Statistical testing summary table created successfully.
Saved to: ../results/tables/statistical_testing_summary.csv


,Model,Baseline MAE,Climate-Enhanced MAE,MAE Difference,p-value,Significant at 5%,Conclusion
0,Linear Regression,1601.21,1578.41,22.80,0.000,True,Significant improvement
1,Random Forest,5107.06,5283.10,-176.03,0.996,False,No improvement
2,XGBoost,9252.48,13214.53,-3962.05,1.000,False,No improvement


### Step 5 Key Takeaways

* Prediction errors were calculated for each baseline and climate-enhanced model using the same 2023–2025 holdout test observations.
* A one-sided paired t-test was used to evaluate whether climate-enhanced models significantly reduced absolute prediction errors compared with their baseline versions.
* Linear Regression showed a statistically significant improvement after adding climate-risk variables, with average absolute error decreasing from 1601.21 to 1578.41.
* Although the Linear Regression improvement was statistically significant, the size of the improvement was modest.
* Random Forest did not show a statistically significant improvement in average absolute error; its climate-enhanced version had a slightly higher MAE than the baseline version.
* XGBoost also did not show improvement; its climate-enhanced version performed worse than the baseline version.
* Overall, the statistical results suggest that climate-risk variables provide limited and model-dependent predictive value under the current lag-informed specification.
* The strong performance of the baseline models, especially Linear Regression, suggests that lagged housing prices dominate short-term county-level housing valuation.
* A no-lag robustness check should be considered to evaluate whether climate-risk variables provide stronger predictive value when lagged housing prices are excluded.



## 6. Phase 3 Summary

Summarize the main findings from model development and validation.


### 6.1 Best-Performing Model

Identify which model achieved the strongest holdout performance.


### 6.2 Climate-Risk Contribution

Summarize whether adding climate-risk variables improved housing valuation performance across model families.


### 6.3 Transition to Phase 4

Briefly explain how Phase 3 results motivate Phase 4, where model interpretation, feature importance, SHAP analysis, and climate-risk explanation will be conducted.
